# IRAS Two Orbit

Investigation of the behavior of the generalized Rayleigh quotient iterations in IRAS.

## Setup: Observation Covariance Matrix $\Sigma$

We construct the observation covariance matrix $\Sigma$ as follows:

1. Choose dimension $n$ of the observations.
2. Randomly generate $\Sigma_x \in \mathbb{R}^{(n-1)\times(n-1)}$, a positive-definite covariance matrix, normalized so that the minimal absolute eigenvalue equals 1.
3. Draw $\sigma^2 \sim \mathrm{Uniform}(0.2,\, 0.7)$.
4. Draw an orthonormal basis $\{v^1, v^2, \ldots, v^n\}$ of $\mathbb{R}^n$ and define
$$
V_\perp := \begin{bmatrix} v^2 & \cdots & v^n \end{bmatrix} \in \mathbb{R}^{n\times(n-1)}.
$$
5. The observation covariance matrix is then:
$$
\Sigma := \sigma^2 v^1 {v^1}^\prime + V_\perp \Sigma_x V_\perp^\prime.
$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from numpy.linalg import eigh, eigvalsh

rng = np.random.default_rng()

# ── Shared display helper ──────────────────────────────────────────────────────
def plot_matrix(ax, M, title, row_labels, col_labels, fmt=".3f", cmap="RdBu_r"):
    """Render matrix M as an annotated heatmap on axes ax."""
    vmax = np.abs(M).max()
    vmin = -vmax if M.min() < 0 else 0
    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(M.shape[1]))
    ax.set_yticks(range(M.shape[0]))
    ax.set_xticklabels(col_labels, fontsize=9)
    ax.set_yticklabels(row_labels, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            val = format(M[i, j], fmt)
            color = "white" if abs(M[i, j]) > 0.6 * vmax else "black"
            ax.text(j, i, val, ha="center", va="center", fontsize=8, color=color)
    return im


def add_eigenvalue_bar(fig, ax, eigvals, title, color="steelblue"):
    """Add a compact eigenvalue bar chart as an inset."""
    inset = ax.inset_axes([1.08, 0.0, 0.35, 1.0])
    inset.barh(range(len(eigvals)), sorted(eigvals), color=color, alpha=0.8)
    inset.set_yticks(range(len(eigvals)))
    inset.set_yticklabels([f"$\\lambda_{{{i+1}}}$" for i in range(len(eigvals))], fontsize=8)
    inset.set_xlabel("eigenvalue", fontsize=7)
    inset.set_title("eigs", fontsize=8)
    inset.tick_params(axis="both", labelsize=7)
    inset.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.5)

## Helper: $\tilde{\Sigma}(\theta,\,\bar\sigma^2,\,\Sigma)$

Given a unit vector $\theta \in \mathbb{R}^n$, a scalar hyper-parameter $\bar\sigma^2 > 0$,
and the observation covariance $\Sigma$, define
$$
s(\theta) := \operatorname{var}(\theta^\prime z) = \theta^\prime\Sigma\,\theta,
$$
and
$$
\tilde{\Sigma}(\theta,\,\bar\sigma^2,\,\Sigma)
:=
s(\theta)\,\theta\theta^\prime
+
\bar\sigma^2\bigl(I_n - \theta\theta^\prime\bigr).
$$
Note that $\theta$ is assumed to be a **unit vector** so that $\theta\theta^\prime$ is the
rank-1 orthogonal projector onto $\operatorname{span}(\theta)$.

In [ ]:
def compute_Sigma_tilde(theta, bar_sigma2, Sigma):
    """
    Compute the approximate covariance matrix Sigma_tilde.

    Parameters
    ----------
    theta      : array_like, shape (n,)  -- unit vector in R^n
    bar_sigma2 : float                   -- hyper-parameter (noise variance in
                                           directions orthogonal to theta)
    Sigma      : array_like, shape (n,n) -- observation covariance matrix

    Returns
    -------
    Sigma_tilde : ndarray, shape (n,n)
        s(theta) * outer(theta, theta) + bar_sigma2 * (I - outer(theta, theta))
        where s(theta) = theta' @ Sigma @ theta.
    s_theta     : float
        The scalar s(theta) = Var(theta' z).
    """
    theta = np.asarray(theta, dtype=float)
    Sigma = np.asarray(Sigma, dtype=float)
    n = len(theta)

    s_theta = float(theta @ Sigma @ theta)          # s(theta) = theta' Sigma theta
    P_theta = np.outer(theta, theta)                # rank-1 projector onto span(theta)
    Sigma_tilde = s_theta * P_theta + bar_sigma2 * (np.eye(n) - P_theta)
    return Sigma_tilde, s_theta

## Helper: single IRAS iteration

Given $\theta^{i-1}$, compute $\theta^i$ as the **dominant eigenvector** of
$\Sigma^{-1}\tilde{\Sigma}(\theta^{i-1})$, i.e. the eigenvector corresponding
to the largest eigenvalue of the generalised eigenproblem
$$
\tilde{\Sigma}(\theta^{i-1})\,v = \lambda\,\Sigma\,v.
$$
The returned vector is normalised to unit length.

In [ ]:
from scipy.linalg import eigh as scipy_eigh

def iras_iteration(theta_prev, bar_sigma2, Sigma):
    """
    Perform one IRAS iteration.

    Parameters
    ----------
    theta_prev : array_like, shape (n,)  -- unit vector from the previous iteration
    bar_sigma2 : float                   -- hyper-parameter
    Sigma      : array_like, shape (n,n) -- observation covariance matrix

    Returns
    -------
    theta_new : ndarray, shape (n,)
        Dominant eigenvector of Sigma^{-1} @ Sigma_tilde(theta_prev),
        normalised to unit length.
    eigenvalue : float
        The corresponding (largest) generalised eigenvalue.
    """
    Sigma_tilde, _ = compute_Sigma_tilde(theta_prev, bar_sigma2, Sigma)

    # Solve the generalised eigenproblem Sigma_tilde v = lambda Sigma v.
    # scipy_eigh returns eigenvalues in ascending order; take the last one.
    eigvals, eigvecs = scipy_eigh(Sigma_tilde, Sigma)
    theta_new = eigvecs[:, -1]                  # dominant eigenvector
    theta_new = theta_new / np.linalg.norm(theta_new)  # ensure unit length
    return theta_new, float(eigvals[-1])


def v1_hat(tm1, tm2, threshold=-0.5):
    """
    Adaptive estimator for v^1 from the last two IRAS iterates.

    Parameters
    ----------
    tm1       : array_like, shape (n,)  -- theta[-1]  (last iterate)
    tm2       : array_like, shape (n,)  -- theta[-2]  (second-to-last)
    threshold : float                   -- dot-product regime boundary (default -0.5)

    Returns
    -------
    v_hat : ndarray, shape (n,)   -- unit-vector estimate of v^1
    dot   : float                 -- dot product tm1 · tm2 (regime indicator)
    """
    dot  = float(tm1 @ tm2)
    diff = tm1 - tm2;  u_m = diff / np.linalg.norm(diff)
    s    = tm1 + tm2;  u_p = s    / np.linalg.norm(s)
    if dot < threshold:            # valid regime: orbit symmetric about v^1
        return u_m, dot
    else:                          # trapped regime: orbit in V_perp
        comb = u_m - u_p
        return comb / np.linalg.norm(comb), dot


def generate_problem(n, rng):
    """
    Generate a full problem instance.

    Parameters
    ----------
    n   : int              -- dimension of observations
    rng : np.random.Generator

    Returns
    -------
    dict with keys:
        Sigma_x    : ndarray (n-1, n-1)  -- latent covariance (eigenvalues in [1, 2])
        eigvals_x  : ndarray (n-1,)      -- sorted eigenvalues of Sigma_x
        Q          : ndarray (n, n)      -- full orthonormal basis; columns are v^1,...,v^n
        v1         : ndarray (n,)        -- first basis vector
        V_perp     : ndarray (n, n-1)    -- remaining basis vectors [v^2,...,v^n]
        sigma2     : float               -- noise variance along v^1 (in [0.2, 0.7])
        bar_sigma2 : float               -- hyper-parameter  d * max|eig(Sigma)|
        d          : float               -- multiplier drawn from Uniform(1.5, 2)
    """
    m = n - 1

    # Sigma_x: eigenvalues in [1, 2]
    eigvals_x = np.sort(rng.uniform(1.0, 2.0, size=m))
    Q_x, _   = np.linalg.qr(rng.standard_normal((m, m)))
    Sigma_x   = Q_x @ np.diag(eigvals_x) @ Q_x.T

    # sigma^2
    sigma2 = float(rng.uniform(0.2, 0.7))

    # Orthonormal basis {v^1, ..., v^n}
    Q, _   = np.linalg.qr(rng.standard_normal((n, n)))
    v1     = Q[:, 0]
    V_perp = Q[:, 1:]

    # bar_sigma^2 = d * max|eig(Sigma)|
    Sigma = compute_Sigma(Sigma_x, v1, V_perp, sigma2)
    #d = float(rng.uniform(1.5, 2.0))
    
    bar_sigma2 = float(rng.uniform(sigma2, (sigma2 + eigvals_x.min())/2)) #d * float(np.max(np.abs(eigvalsh(Sigma))))

    return dict(
        Sigma_x=Sigma_x, eigvals_x=eigvals_x,
        Q=Q, v1=v1, V_perp=V_perp,
        sigma2=sigma2, bar_sigma2=bar_sigma2, #d=d,
    )


def compute_Sigma(Sigma_x, v1, V_perp, sigma2):
    """
    Construct the observation covariance matrix.

    Sigma = sigma2 * outer(v1, v1) + V_perp @ Sigma_x @ V_perp.T
    """
    return sigma2 * np.outer(v1, v1) + V_perp @ Sigma_x @ V_perp.T

In [ ]:
# --- User parameter ---
n = 5  # dimension of observations

In [ ]:
# ── Generate problem instance ─────────────────────────────────────────────────
prob = generate_problem(n, rng)
Sigma = compute_Sigma(prob['Sigma_x'], prob['v1'], prob['V_perp'], prob['sigma2'])

# Unpack for convenience
m             = n - 1
Sigma_x       = prob['Sigma_x']
eigvals_x_norm = prob['eigvals_x']   # sorted eigenvalues of Sigma_x
Q             = prob['Q']            # full orthonormal basis matrix
v1            = prob['v1']
V_perp        = prob['V_perp']
sigma2        = prob['sigma2']
bar_sigma2    = prob['bar_sigma2']
#d             = prob['d']
eigvals_Sigma = eigvalsh(Sigma)

### Step 1 — Generate $\Sigma_x$

In [ ]:
# ---------------------------------------------------------------
# Sigma_x was generated by generate_problem() — verify eigenvalue range
# ---------------------------------------------------------------
assert np.all(eigvals_x_norm >= 1.0 - 1e-10) and np.all(eigvals_x_norm <= 2.0 + 1e-10)

In [ ]:
idx = [f"$x_{{{i+1}}}$" for i in range(m)]

fig, ax = plt.subplots(figsize=(4.5, 3.5))
fig.subplots_adjust(right=0.72)
im = plot_matrix(ax, Sigma_x, r"$\Sigma_x$  —  latent covariance", idx, idx)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
add_eigenvalue_bar(fig, ax, eigvals_x_norm, r"$\Sigma_x$ eigenvalues", color="teal")
plt.suptitle(
    f"$\\Sigma_x \\in \\mathbb{{R}}^{{{m}\\times{m}}}$, "
    f"min$|\\lambda|$ = {np.min(np.abs(eigvals_x_norm)):.3f}",
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.show()

# ── Eigenvalue table ──────────────────────────────────────────────────────────
sorted_eigs = np.sort(eigvals_x_norm)  # ascending (eigvalsh already returns sorted)
df_eigs = pd.DataFrame(
    {"Eigenvalue": sorted_eigs},
    index=[f"$\\lambda_{{{i+1}}}$" for i in range(m)]
)
display(
    df_eigs.style
    .set_caption(r"Eigenvalues of $\Sigma_x$ (ascending)")
    .format("{:.6f}")
    .bar(subset=["Eigenvalue"], color="#5fba7d", vmin=0)
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("padding-bottom", "6px")]},
        {"selector": "th",
         "props": [("font-size", "11px"), ("text-align", "center"),
                   ("background-color", "#f0f0f0")]},
        {"selector": "td",
         "props": [("font-size", "11px"), ("text-align", "right"),
                   ("min-width", "120px")]},
    ])
)

### Step 2 — Draw $\sigma^2$ and orthonormal basis $\{v^1,\ldots,v^n\}$

In [ ]:
# sigma2 and orthonormal basis were generated by generate_problem()
print(f"σ²  =  {sigma2:.6f}")

In [ ]:
# ── Display orthonormal basis as a styled DataFrame ───────────────────────────
col_names = [f"$v^{{{i+1}}}$" for i in range(n)]
row_names = [f"$e_{{{i+1}}}$" for i in range(n)]

df_basis = pd.DataFrame(Q, index=row_names, columns=col_names)

styled = (
    df_basis
    .style
    .set_caption(
        r"Orthonormal basis  $\{v^1,\ldots,v^n\}$  — columns of $Q$"
    )
    .format("{:.4f}")
    .background_gradient(cmap="RdBu_r", vmin=-1, vmax=1, axis=None)
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"),
                   ("padding-bottom", "6px")]},
        {"selector": "th",
         "props": [("font-size", "11px"), ("text-align", "center"),
                   ("background-color", "#f0f0f0")]},
        {"selector": "td",
         "props": [("font-size", "11px"), ("text-align", "center"),
                   ("min-width", "70px")]},
    ])
)

# Highlight v^1 column border
styled = styled.set_properties(subset=["$v^{1}$"],
                               **{"border-left": "3px solid #e05c2a",
                                  "border-right": "3px solid #e05c2a"})
display(styled)

In [ ]:
# ── Quick orthonormality sanity check ─────────────────────────────────────────
GtG = Q.T @ Q
df_check = pd.DataFrame(GtG, index=col_names, columns=col_names)
df_check.style \
    .set_caption(r"$Q^\top Q$ — should be the identity") \
    .format("{:.6f}") \
    .background_gradient(cmap="Greens", vmin=0, vmax=1, axis=None) \
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "12px"), ("font-weight", "bold"),
                   ("padding-bottom", "5px")]},
        {"selector": "td",
         "props": [("font-size", "10px"), ("text-align", "center"),
                   ("min-width", "80px")]},
    ])

### Step 3 — Construct $\Sigma$

In [ ]:
# Sigma was built by compute_Sigma() in the problem-setup cell — verify properties
assert np.allclose(Sigma, Sigma.T), "Sigma not symmetric"
assert np.all(eigvals_Sigma > 0), "Sigma not positive definite"

In [ ]:
z_idx = [f"$z_{{{i+1}}}$" for i in range(n)]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8),
                         gridspec_kw={"width_ratios": [1, 1]})
fig.subplots_adjust(right=0.82, wspace=0.55)

# Left: Sigma_x
im0 = plot_matrix(axes[0], Sigma_x,
                  r"$\Sigma_x$  $(latent)$", idx, idx)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
add_eigenvalue_bar(fig, axes[0], eigvals_x_norm, "", color="teal")

# Right: Sigma
im1 = plot_matrix(axes[1], Sigma,
                  r"$\Sigma$  $(observation)$", z_idx, z_idx)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
add_eigenvalue_bar(fig, axes[1], eigvals_Sigma, "", color="steelblue")

fig.suptitle(
    rf"$n={n}$, $\sigma^2={sigma2:.4f}$     "
    r"$\Sigma = \sigma^2 v^1{v^1}^\prime + V_\perp\,\Sigma_x\,V_\perp^\prime$",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

print(f"Σ eigenvalues : {np.round(eigvals_Sigma, 4)}")
print(f"σ²            : {sigma2:.4f}  (should match smallest Σ eigenvalue)")

### Step 4 — Draw $d$ and set $\bar{\sigma}^2$

Draw $d \sim \mathrm{Uniform}(1.5,\,2)$ and set
$$
\bar{\sigma}^2 := d \cdot \max_i |\lambda_i(\Sigma)|,
$$
where $\lambda_i(\Sigma)$ are the eigenvalues of $\Sigma$.

In [ ]:
# d and bar_sigma2 were generated by generate_problem()
#print(f"d            = {d:.6f}")
print(f"max|λ(Σ)|    = {np.max(np.abs(eigvals_Sigma)):.6f}")
print(f"σ̄²           = {bar_sigma2:.6f}")

## IRAS iterations: convergence experiment

All $N$ simulations use the **same** problem instance $(\Sigma,\,\bar\sigma^2)$ generated
above, but each starts from a **different random unit vector** $\theta^0 \in \mathbb{R}^n$.
After each iteration we record the **angle** between $\theta^i$ and the target $v^1$:
$$
\phi^i := \arccos\!\bigl(|{\theta^i}^\prime v^1|\bigr) \in [0,\,\pi/2].
$$
(The absolute value removes the sign ambiguity of eigenvectors.)
The plot therefore shows how different initial conditions converge — or fail to converge —
under the same IRAS dynamics.

In [ ]:
N = 20     # number of independent simulations
L = 6  # number of IRAS iterations per simulation

# ── Run simulations ───────────────────────────────────────────────────────────
angles = np.zeros((N, L + 1))  # angles[sim, i] = angle at iteration i

from tqdm.notebook import tqdm

for sim in tqdm(range(N), desc="Running simulations"):
    # Random initial unit vector theta^0
    theta = rng.standard_normal(n)
    theta = theta / np.linalg.norm(theta)

    # Record angle at i=0
    angles[sim, 0] = np.arccos(np.clip(np.abs(theta @ v1), 0.0, 1.0))

    for i in range(1, L + 1):
        theta, _ = iras_iteration(theta, bar_sigma2, Sigma)
        angles[sim, i] = np.arccos(np.clip(np.abs(theta @ v1), 0.0, 1.0))


angles_deg = np.degrees(angles)  # convert to degrees for readability

# ── Interactive 3×1 subplot (plotly) ─────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"
]

# Even iterations: θ^{2k} for k = 0, 1, 2, ...
even_idx = np.arange(0, L + 1, 2)           # 0, 2, 4, ...
k_even   = np.arange(len(even_idx))          # 0, 1, 2, ...

# Odd iterations: θ^{2k+1} for k = 0, 1, 2, ...
odd_idx  = np.arange(1, L + 1, 2)           # 1, 3, 5, ...
k_odd    = np.arange(len(odd_idx))           # 0, 1, 2, ...

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.10,
    subplot_titles=[
        "All iterations  θⁱ",
        "Even sub-sequence  θ^{2k}",
        "Odd sub-sequence  θ^{2k+1}",
    ],
)

for sim in range(N):
    c = colors[sim % len(colors)]
    show_legend = True   # show in legend only once (first row)

    # Row 1: all iterations
    fig.add_trace(
        go.Scatter(x=np.arange(L + 1), y=angles_deg[sim],
                   mode="lines", name=f"sim {sim+1}",
                   line=dict(color=c, width=1.2), opacity=0.85,
                   legendgroup=f"sim{sim+1}", showlegend=True),
        row=1, col=1,
    )
    # Row 2: even iterations
    fig.add_trace(
        go.Scatter(x=k_even, y=angles_deg[sim, even_idx],
                   mode="lines", name=f"sim {sim+1}",
                   line=dict(color=c, width=1.2), opacity=0.85,
                   legendgroup=f"sim{sim+1}", showlegend=False),
        row=2, col=1,
    )
    # Row 3: odd iterations
    fig.add_trace(
        go.Scatter(x=k_odd, y=angles_deg[sim, odd_idx],
                   mode="lines", name=f"sim {sim+1}",
                   line=dict(color=c, width=1.2), opacity=0.85,
                   legendgroup=f"sim{sim+1}", showlegend=False),
        row=3, col=1,
    )

fig.update_xaxes(title_text="iteration  i",              showgrid=True, gridcolor="lightgrey", row=1, col=1)
fig.update_xaxes(title_text="k  (iteration = 2k)",       showgrid=True, gridcolor="lightgrey", row=2, col=1)
fig.update_xaxes(title_text="k  (iteration = 2k+1)",     showgrid=True, gridcolor="lightgrey", row=3, col=1)
fig.update_yaxes(title_text="∠(θⁱ, v¹)  (degrees)", showgrid=True, gridcolor="lightgrey")

fig.update_layout(
    title=dict(
        text=f"IRAS convergence  (n={n}, N={N} simulations, L={L} iterations)",
        font=dict(size=14),
    ),
    legend=dict(x=1.01, y=1, bordercolor="grey", borderwidth=1),
    hovermode="x unified",
    plot_bgcolor="white",
    width=900,
    height=900,
)
fig.show()

print(f"Final angles (degrees) after {L} iterations:")
for sim in range(N):
    print(f"  sim {sim+1:2d}: {angles_deg[sim, -1]:.6f}°")

In [ ]:
# ── Large-scale validation: angles of θ[-1] vs v¹ over many problems & starts ─
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from tqdm.notebook import tqdm

N_problems = 1000
N_starts   = 1000
L_big      = 12

all_angles = []  # will hold N_problems * N_starts values (degrees)

for prob_idx in tqdm(range(N_problems), desc="Problems"):
    prob_i  = generate_problem(n, rng)
    Sigma_i = compute_Sigma(prob_i['Sigma_x'], prob_i['v1'], prob_i['V_perp'], prob_i['sigma2'])
    v1_i    = prob_i['v1']
    bs2_i   = prob_i['bar_sigma2']

    for _ in range(N_starts):
        theta = rng.standard_normal(n)
        theta /= np.linalg.norm(theta)
        for __ in range(L_big):
            theta, _ = iras_iteration(theta, bs2_i, Sigma_i)
        ang = np.degrees(np.arccos(np.clip(np.abs(theta @ v1_i), 0.0, 1.0)))
        all_angles.append(ang)

all_angles = np.array(all_angles)

print(f"Total samples : {len(all_angles):,}")
print(f"mean  = {all_angles.mean():.5f}°")
print(f"median= {np.median(all_angles):.5f}°")
print(f"max   = {all_angles.max():.5f}°")
print(f"<0.1° : {(all_angles < 0.1).sum():,} / {len(all_angles):,} "
      f"({100*(all_angles<0.1).mean():.2f}%)")

# ── KDE ───────────────────────────────────────────────────────────────────────
kde      = gaussian_kde(all_angles, bw_method='scott')
x_grid   = np.linspace(0, all_angles.max() * 1.05, 500)
kde_vals = kde(x_grid)

# ── Empirical CDF ─────────────────────────────────────────────────────────────
sorted_ang = np.sort(all_angles)
cdf_vals   = np.arange(1, len(sorted_ang) + 1) / len(sorted_ang)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=1,
    vertical_spacing=0.12,
    subplot_titles=[
        f"PDF of ∠(θ<sup>[-1]</sup>, v¹)  —  {N_problems}×{N_starts} samples",
        f"CDF of ∠(θ<sup>[-1]</sup>, v¹)",
    ],
)

# --- PDF: histogram ---
fig.add_trace(
    go.Histogram(
        x=all_angles,
        histnorm='probability density',
        nbinsx=120,
        name="histogram",
        marker_color="steelblue",
        opacity=0.55,
    ),
    row=1, col=1,
)
# --- PDF: KDE ---
fig.add_trace(
    go.Scatter(
        x=x_grid, y=kde_vals,
        mode="lines",
        name="KDE",
        line=dict(color="darkorange", width=2),
    ),
    row=1, col=1,
)

# --- CDF ---
fig.add_trace(
    go.Scatter(
        x=sorted_ang, y=cdf_vals,
        mode="lines",
        name="empirical CDF",
        line=dict(color="steelblue", width=1.5),
        showlegend=True,
    ),
    row=2, col=1,
)

fig.update_xaxes(title_text="∠(θ[-1], v¹)  (degrees)", showgrid=True, gridcolor="lightgrey", row=1, col=1)
fig.update_xaxes(title_text="∠(θ[-1], v¹)  (degrees)", showgrid=True, gridcolor="lightgrey", row=2, col=1)
fig.update_yaxes(title_text="density",      showgrid=True, gridcolor="lightgrey", row=1, col=1)
fig.update_yaxes(title_text="CDF",          showgrid=True, gridcolor="lightgrey", row=2, col=1)

fig.update_layout(
    title=dict(
        text=(f"IRAS convergence distribution  "
              f"(n={n}, {N_problems} problems × {N_starts} starts, L={L_big} iterations)"),
        font=dict(size=13),
    ),
    plot_bgcolor="white",
    legend=dict(x=0.75, y=0.95, bordercolor="grey", borderwidth=1),
    width=850, height=700,
)
fig.show()